In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.spatial.distance import pdist

from dtaidistance import dtw


## Load Generated Samples

In [ ]:
INPUT_PATH = Path('data/generated/generated_returns.json')

with open(INPUT_PATH) as f:
    data = json.load(f)

conditions = data['conditions']
samples_np = np.array(data['samples'], dtype=np.float64)  # (N, T)

N, T = samples_np.shape
n_pairs = N * (N - 1) // 2

print("Conditions:")
for k, v in conditions.items():
    print(f"  {k}: {v}")
print(f"\nSamples: {N} sequences of length {T}")
print(f"Pairs to evaluate: {n_pairs:,}")

## Euclidean Distances

In [ ]:
print("Computing pairwise Euclidean distances...")
euc_dists = pdist(samples_np, metric='euclidean')  # shape (n_pairs,)
print(f"Done. Shape: {euc_dists.shape}")

## DTW Distances

In [ ]:
print("Computing pairwise DTW distances (this may take a minute)...")
dtw_matrix = dtw.distance_matrix_fast(samples_np)   # (N, N) symmetric, diag=0
dtw_dists = dtw_matrix[np.triu_indices(N, k=1)]     # upper triangle → (n_pairs,)
print(f"Done. Shape: {dtw_dists.shape}")

## Summary Statistics

In [ ]:
cond_str = f"trend={conditions['trend']}, realized_vol={conditions['realized_vol']}"

print(f"Condition: {cond_str}")
print(f"{'Metric':<12} {'Mean':>10} {'Std':>10} {'Median':>10} {'Min':>10} {'Max':>10}")
print("-" * 62)
for name, dists in [("Euclidean", euc_dists), ("DTW", dtw_dists)]:
    print(
        f"{name:<12} "
        f"{dists.mean():>10.4f} "
        f"{dists.std():>10.4f} "
        f"{np.median(dists):>10.4f} "
        f"{dists.min():>10.4f} "
        f"{dists.max():>10.4f}"
    )